In [1]:
# install packages
%pip install -q nltk transformers scikit-learn matplotlib torch numpy

# Import libraries
import nltk
import numpy as np
import torch

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import cross_val_score
from transformers import BertModel, BertTokenizer

# Download NLTK treebank dataset
nltk.download('treebank', quiet=True)
from nltk.corpus import treebank

Note: you may need to restart the kernel to use updated packages.


In [2]:
# each sentence is a list of (word, pos_tag) tuples
sentences = treebank.tagged_sents()

# first sentence from the treebank corpus
first_sentence = sentences[0]
print(first_sentence)

# separate the words and the corresponding POS tags
words = []
pos_order = []
for word, pos in first_sentence:
    words.append(word)
    pos_order.append(pos)

# printing sentence and order of POS tags --> visualization of how the data looks like
print("Sentence:", " ".join(words))
print("POS Order:", " ".join(pos_order))

[('Pierre', 'NNP'), ('Vinken', 'NNP'), (',', ','), ('61', 'CD'), ('years', 'NNS'), ('old', 'JJ'), (',', ','), ('will', 'MD'), ('join', 'VB'), ('the', 'DT'), ('board', 'NN'), ('as', 'IN'), ('a', 'DT'), ('nonexecutive', 'JJ'), ('director', 'NN'), ('Nov.', 'NNP'), ('29', 'CD'), ('.', '.')]
Sentence: Pierre Vinken , 61 years old , will join the board as a nonexecutive director Nov. 29 .
POS Order: NNP NNP , CD NNS JJ , MD VB DT NN IN DT JJ NN NNP CD .


In [3]:
# Create sentence-level data
sentences_text = [" ".join([word for word, _ in sentence]) for sentence in sentences]
sentence_lengths = [len(sentence) for sentence in sentences]

# printing to visualize the structure
print("Sample sentence length data:")
for i in range(3):  # Show first 3 examples
    print(f"Sentence {i+1}: {sentences_text[i]}")
    print(f"Length: {sentence_lengths[i]}")
    print()

Sample sentence length data:
Sentence 1: Pierre Vinken , 61 years old , will join the board as a nonexecutive director Nov. 29 .
Length: 18

Sentence 2: Mr. Vinken is chairman of Elsevier N.V. , the Dutch publishing group .
Length: 13

Sentence 3: Rudolph Agnew , 55 years old and former chairman of Consolidated Gold Fields PLC , was named *-1 a nonexecutive director of this British industrial conglomerate .
Length: 27



In [4]:
from sentence_transformers import SentenceTransformer

# Create MiniLM embeddings for each sentence
def get_minilm_embeddings(texts):
    # Load the model
    model = SentenceTransformer('all-MiniLM-L6-v2')
    
    # Process in batches to avoid memory issues
    batch_size = 32
    embeddings = []
    
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        # Get embeddings for the batch
        batch_embeddings = model.encode(batch_texts)
        embeddings.extend(batch_embeddings)
    
    return np.array(embeddings)

# Get MiniLM embeddings for sentences
print("Generating MiniLM-L6 embeddings for sentences...")
X_sentence_features = get_minilm_embeddings(sentences_text)
print(f"Sentence MiniLM-L6 embeddings shape: {X_sentence_features.shape}")

Generating MiniLM-L6 embeddings for sentences...
Sentence MiniLM-L6 embeddings shape: (3914, 384)


In [5]:
def train_linear_model(features, labels):
    """
    Train linear regression model to predict sentence length.
    
    Parameters:
      features (np.ndarray): Feature matrix of shape (num_samples, num_features).
      labels (np.ndarray): Sentence length labels.
      
    Returns:
      model: Trained linear regression model.
      metrics: Dictionary containing evaluation metrics.
    """
    # splitting into training and test sets
    X_train, X_test, y_train, y_test = train_test_split(
        features, labels, test_size=0.2, random_state=42
    )
    
    # Linear regression model
    model = LinearRegression()
    model.fit(X_train, y_train)
    
    # Predictions on the test set
    predictions = model.predict(X_test)
    
    # evaluation
    metrics = {
        "mean_squared_error": mean_squared_error(y_test, predictions),
        "mean_absolute_error": mean_absolute_error(y_test, predictions),
        "r2_score": r2_score(y_test, predictions),
    }
    
    return model, metrics, X_test, y_test

In [6]:
print("Training linear regression model on sentence embeddings...")
linear_model, linear_metrics, X_test, y_test = train_linear_model(X_sentence_features, np.array(sentence_lengths))
print("Linear Regression Evaluation Metrics for Sentence Length Prediction:")
print(f"Mean Squared Error: {linear_metrics['mean_squared_error']:.4f}")
print(f"Mean Absolute Error: {linear_metrics['mean_absolute_error']:.4f}")
print(f"R² Score: {linear_metrics['r2_score']:.4f}")

Training linear regression model on sentence embeddings...
Linear Regression Evaluation Metrics for Sentence Length Prediction:
Mean Squared Error: 63.4912
Mean Absolute Error: 5.7951
R² Score: 0.5646


In [7]:
import ipywidgets as widgets
from IPython.display import display

def display_prediction(index):
    if index < 0 or index >= len(X_test):
        print("Index out of range. Please select a valid index.")
        return

    # Get prediction from the linear regression model
    prediction = linear_model.predict([X_test[index]])[0]
    actual = y_test[index]

    try:
        original_sentence = sentences_text[index]
        print(f"Original Sentence: {original_sentence}")
    except Exception:
        print("Original sentence text not available")
    
    print(f"Sentence Index: {index}")
    print(f"Predicted Sentence Length: {prediction:.2f}")
    print(f"Actual Sentence Length: {actual}")
    print(f"Error: {abs(prediction - actual):.2f}")

slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(X_test) - 1,
    step=1,
    description="Sentence Index:",
    continuous_update=False
)

interactive_widget = widgets.interactive(display_prediction, index=slider)
display(interactive_widget)

interactive(children=(IntSlider(value=0, continuous_update=False, description='Sentence Index:', max=782), Out…